# 01 — Exploratory Data Analysis & Cleaning
**Project:** Supply Chain Intelligence System  
**Dataset:** Olist Brazilian E-Commerce  
**Goal:** Understand the data, clean it, and produce a master table for downstream KPI calculations.

---
## Contents
1. Load all datasets
2. Initial inspection
3. Data quality checks
4. Cleaning + feature engineering
5. Merge into master table
6. Export to `data/processed/`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print('Libraries loaded successfully')

## 1. Load All Datasets

In [ ]:
DATA_PATH = '../data/raw/'

orders     = pd.read_csv(DATA_PATH + 'olist_orders_dataset.csv')
items      = pd.read_csv(DATA_PATH + 'olist_order_items_dataset.csv')
products   = pd.read_csv(DATA_PATH + 'olist_products_dataset.csv')
sellers    = pd.read_csv(DATA_PATH + 'olist_sellers_dataset.csv')
customers  = pd.read_csv(DATA_PATH + 'olist_customers_dataset.csv')
reviews    = pd.read_csv(DATA_PATH + 'olist_order_reviews_dataset.csv')
payments   = pd.read_csv(DATA_PATH + 'olist_order_payments_dataset.csv')
category_translation = pd.read_csv(DATA_PATH + 'product_category_name_translation.csv')

print(f'Orders:    {orders.shape}')
print(f'Items:     {items.shape}')
print(f'Products:  {products.shape}')
print(f'Sellers:   {sellers.shape}')
print(f'Customers: {customers.shape}')
print(f'Reviews:   {reviews.shape}')
print(f'Payments:  {payments.shape}')

## 2. Initial Inspection

In [ ]:
print('=== ORDERS ===')
display(orders.head(3))
print('\nOrder statuses:')
print(orders['order_status'].value_counts())

In [ ]:
print('=== ORDER ITEMS ===')
display(items.head(3))
print(f'\nAvg items per order: {items.groupby("order_id")["order_item_id"].max().mean():.2f}')

## 3. Data Quality Checks

In [ ]:
print('=== NULL CHECK ===')
nulls = orders.isnull().sum()
print(nulls[nulls > 0])
print(f'\nTotal orders: {len(orders)}')
print(f'Orders with null delivery date: {orders["order_delivered_customer_date"].isna().sum()}')
print(f'(These are non-delivered orders — expected)')

In [ ]:
# Check date columns
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

print('Date range of orders:')
print(f'  First order: {orders["order_purchase_timestamp"].min()}')
print(f'  Last order:  {orders["order_purchase_timestamp"].max()}')

## 4. Cleaning + Feature Engineering

In [ ]:
# Keep only delivered orders for supply chain KPIs
delivered = orders[orders['order_status'] == 'delivered'].copy()
print(f'Delivered orders: {len(delivered)} of {len(orders)} ({len(delivered)/len(orders)*100:.1f}%)')

# --- Key supply chain metrics ---

# 1. Fulfillment latency (days from purchase to delivery)
delivered['fulfillment_days'] = (
    delivered['order_delivered_customer_date'] - delivered['order_purchase_timestamp']
).dt.days

# 2. On-time delivery flag
delivered['on_time'] = (
    delivered['order_delivered_customer_date'] <= delivered['order_estimated_delivery_date']
).astype(int)

# 3. Delivery gap (positive = late, negative = early)
delivered['delivery_gap_days'] = (
    delivered['order_delivered_customer_date'] - delivered['order_estimated_delivery_date']
).dt.days

# 4. Time to carrier handoff
delivered['processing_days'] = (
    delivered['order_delivered_carrier_date'] - delivered['order_approved_at']
).dt.days

# 5. Purchase month/year for time series
delivered['purchase_month'] = delivered['order_purchase_timestamp'].dt.to_period('M')
delivered['purchase_year']  = delivered['order_purchase_timestamp'].dt.year

print('\nFulfillment latency stats (days):')
print(delivered['fulfillment_days'].describe().round(1))
print(f'\nOverall on-time rate: {delivered["on_time"].mean()*100:.1f}%')

In [ ]:
# Remove outliers: orders with negative or extreme latency
before = len(delivered)
delivered = delivered[
    (delivered['fulfillment_days'] >= 0) &
    (delivered['fulfillment_days'] <= 120)
]
print(f'Removed {before - len(delivered)} outlier orders ({(before-len(delivered))/before*100:.1f}%)')

In [ ]:
# Translate product categories
products = products.merge(category_translation, on='product_category_name', how='left')
products['category_en'] = products['product_category_name_english'].fillna(products['product_category_name'])

print('Top 10 product categories:')
print(products['category_en'].value_counts().head(10))

## 5. Build Master Table

In [ ]:
# Merge everything into one analysis table
master = delivered.merge(items, on='order_id', how='left')
master = master.merge(products[['product_id','category_en','product_weight_g']], on='product_id', how='left')
master = master.merge(sellers[['seller_id','seller_state','seller_city']], on='seller_id', how='left')
master = master.merge(customers[['customer_id','customer_state','customer_city']], on='customer_id', how='left')

# Add review scores
review_agg = reviews.groupby('order_id')['review_score'].mean().reset_index()
master = master.merge(review_agg, on='order_id', how='left')

# Add payment totals
payment_agg = payments.groupby('order_id')['payment_value'].sum().reset_index()
master = master.merge(payment_agg, on='order_id', how='left')

print(f'Master table shape: {master.shape}')
display(master.head(3))

## 6. EDA Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Supply Chain Health Overview — Olist Brazil', fontsize=16, fontweight='bold')

# 1. Fulfillment latency distribution
axes[0,0].hist(master['fulfillment_days'].dropna(), bins=40, color='steelblue', edgecolor='white')
axes[0,0].axvline(master['fulfillment_days'].median(), color='red', linestyle='--', label=f'Median: {master["fulfillment_days"].median():.0f}d')
axes[0,0].axvline(master['fulfillment_days'].quantile(0.95), color='orange', linestyle='--', label=f'p95: {master["fulfillment_days"].quantile(0.95):.0f}d')
axes[0,0].set_title('Fulfillment Latency Distribution')
axes[0,0].set_xlabel('Days from purchase to delivery')
axes[0,0].legend()

# 2. On-time rate by state
state_ontime = master.groupby('customer_state')['on_time'].mean().sort_values(ascending=True).tail(15)
axes[0,1].barh(state_ontime.index, state_ontime.values * 100, color='teal')
axes[0,1].axvline(80, color='red', linestyle='--', label='80% threshold')
axes[0,1].set_title('On-Time Delivery Rate by Customer State')
axes[0,1].set_xlabel('On-time rate (%)')
axes[0,1].legend()

# 3. Monthly order volume
monthly = master.groupby('purchase_month')['order_id'].nunique()
monthly.plot(ax=axes[1,0], color='steelblue', marker='o', linewidth=2)
axes[1,0].set_title('Monthly Order Volume')
axes[1,0].set_xlabel('Month')
axes[1,0].set_ylabel('# Unique Orders')
axes[1,0].tick_params(axis='x', rotation=45)

# 4. Review score vs delivery delay
delay_bins = pd.cut(master['delivery_gap_days'], bins=[-60, -14, -7, 0, 7, 14, 60],
                    labels=['Early 2w+','Early 1w','On-time','1w late','2w late','2w+ late'])
review_by_delay = master.groupby(delay_bins)['review_score'].mean()
review_by_delay.plot(kind='bar', ax=axes[1,1], color='coral', edgecolor='white')
axes[1,1].set_title('Avg Review Score by Delivery Timing')
axes[1,1].set_xlabel('Delivery vs estimated date')
axes[1,1].set_ylabel('Avg review score (1-5)')
axes[1,1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../reports/eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to reports/eda_overview.png')

## 7. Export Cleaned Data

In [ ]:
# Export master table
master.to_csv('../data/processed/master_orders.csv', index=False)

# Export seller-level summary (for Tableau)
seller_summary = master.groupby('seller_id').agg(
    total_orders     = ('order_id', 'nunique'),
    on_time_rate     = ('on_time', 'mean'),
    avg_latency_days = ('fulfillment_days', 'mean'),
    avg_review       = ('review_score', 'mean'),
    total_revenue    = ('payment_value', 'sum'),
    seller_state     = ('seller_state', 'first')
).reset_index()
seller_summary.to_csv('../data/processed/seller_scorecard.csv', index=False)

# Export category-level summary (for Tableau)
category_summary = master.groupby('category_en').agg(
    total_orders     = ('order_id', 'nunique'),
    on_time_rate     = ('on_time', 'mean'),
    avg_latency_days = ('fulfillment_days', 'mean'),
    avg_review       = ('review_score', 'mean'),
    total_revenue    = ('payment_value', 'sum')
).reset_index()
category_summary.to_csv('../data/processed/category_summary.csv', index=False)

print('Exported:')
print('  data/processed/master_orders.csv')
print('  data/processed/seller_scorecard.csv')
print('  data/processed/category_summary.csv')
print(f'\nMaster table: {master.shape[0]:,} rows x {master.shape[1]} columns')